# Review Sentiment Experiment: Does the Reviews Data Help?

Standalone experiment -- does **not** touch `main.py` / `notebooks/pipeline.ipynb`.

`main.py --with-sentiment` currently reads only the *first 50,000 rows* of
`reviews.csv.gz` (1.19M rows total), which covers just **474 of ~40,000
listings** -- everything else silently gets a neutral placeholder score. This
notebook fixes that (processes the full reviews file, ~33.5k listings covered)
and tests two variants against the current pipeline's baseline (numeric +
encoded categoricals, XGBoost R^2 = 65.15% on this split):

- **+ basic sentiment**: single average VADER compound score per listing
- **+ aspect sentiment**: per-listing scores for host quality, cleanliness,
  location/safety, and amenities (keyword-gated sentences), plus review count

All three variants share the exact same train/test split.


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import CLEANED_TARGET_PATH, RANDOM_STATE, RAW_LISTINGS_PATH, TARGET_COLUMN, TEST_SIZE
from src.data.clean import clean_missing_target
from src.data.load import load_reviews
from src.evaluate import print_metrics_log_target, regression_metrics_log_target
from src.features.build_features import build_feature_matrix
from src.features.categorical import fit_categorical_encoders, transform_categorical_features
from src.features.sentiment import compute_aspect_sentiment, compute_review_sentiment
from src.models.tree_models import train_xgboost


## Step 1: Build the same feature set as the main pipeline (numeric + categorical)

In [2]:
clean_missing_target(RAW_LISTINGS_PATH, CLEANED_TARGET_PATH, target_column=TARGET_COLUMN)
df = pd.read_csv(CLEANED_TARGET_PATH)
df = build_feature_matrix(df)
df.shape


Reading /Users/jnanadeep/Desktop/airbnb-price-prediction/data/raw/listings.csv...



--- Processing Summary ---
Initial row count:          40769
Rows with missing 'price': 953 (Removed)
Remaining row count:        39816
Cleaned data saved to:      /Users/jnanadeep/Desktop/airbnb-price-prediction/data/processed/airbnb_rio_cleaned_target.csv



Dropping columns with >90.0% missing values:
 - neighborhood_overview
 - host_since
 - host_response_time
 - host_response_rate
 - host_acceptance_rate
 - host_thumbnail_url
 - host_neighbourhood
 - host_total_listings_count
 - host_verifications
 - neighbourhood
 - neighbourhood_group_cleansed
 - calendar_updated
 - license
 - instant_bookable

Imputed 25 numerical columns using their median value.
--- Running Feature Selection ---
Original unique columns: 76
Filtered columns retained: 24
--- Running Dynamic Outlier Removal ---
Calculated 99.0th percentile threshold: $6,999.13
Removed 399 listing(s) outside the range [$10 - $6,999.13].
New maximum price in dataset: $6,990.00


(39417, 27)

## Step 2: Score the *full* reviews file (not just the first 50k rows)\n\nThis takes a couple of minutes -- VADER scores ~1.19M review rows.

In [3]:
reviews = load_reviews()
print(f"Loaded {len(reviews):,} review rows covering {reviews['listing_id'].nunique():,} listings")

basic_sentiment = compute_review_sentiment(reviews)
aspect_sentiment = compute_aspect_sentiment(reviews)

print("Basic sentiment covers", basic_sentiment.shape[0], "listings")
print("Aspect sentiment covers", aspect_sentiment.shape[0], "listings")


Loaded 1,187,985 review rows covering 33,518 listings


Basic sentiment covers 33518 listings
Aspect sentiment covers 33518 listings


## Step 3: Build the three feature variants

In [4]:
df_basic = df.merge(basic_sentiment, left_on="id", right_on="listing_id", how="left").drop(columns=["listing_id"])
df_basic["sentiment_score"] = df_basic["sentiment_score"].fillna(0)

aspect_score_cols = [c for c in aspect_sentiment.columns if c not in ("listing_id",)]
df_aspect = df.merge(aspect_sentiment, left_on="id", right_on="listing_id", how="left").drop(columns=["listing_id"])
df_aspect[aspect_score_cols] = df_aspect[aspect_score_cols].fillna(0)

variants = {
    "A: baseline (numeric + categorical)": df,
    "B: + basic sentiment": df_basic,
    "C: + aspect sentiment": df_aspect,
}


## Step 4: Train XGBoost on each variant (same split, same target)

In [5]:
def evaluate_variant(name, variant_df):
    y_log = np.log1p(variant_df[TARGET_COLUMN])
    train_df, test_df, y_train_log, y_test_log = train_test_split(
        variant_df, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    numeric_cols = variant_df.drop(columns=[TARGET_COLUMN, "id"], errors="ignore").select_dtypes(include=[np.number]).columns

    encoders = fit_categorical_encoders(train_df)
    X_train = pd.concat([train_df[numeric_cols], transform_categorical_features(train_df, encoders)], axis=1)
    X_test = pd.concat([test_df[numeric_cols], transform_categorical_features(test_df, encoders)], axis=1)

    model = train_xgboost(X_train, y_train_log)
    preds_log = model.predict(X_test)
    metrics = print_metrics_log_target(name, y_test_log, preds_log)
    metrics["n_features"] = X_train.shape[1]
    return metrics, model, X_train.columns


results = {}
models = {}
for name, variant_df in variants.items():
    metrics, model, cols = evaluate_variant(name, variant_df)
    results[name] = metrics
    models[name] = (model, cols)


A: baseline (numeric + categorical) | R^2 Score: 65.15% | MAE: $248.73 | RMSE: $502.66


B: + basic sentiment           | R^2 Score: 65.77% | MAE: $245.99 | RMSE: $499.84


C: + aspect sentiment          | R^2 Score: 66.00% | MAE: $245.14 | RMSE: $498.81


## Step 5: Summary

In [6]:
comparison = pd.DataFrame(results).T[["n_features", "mae", "rmse", "r2"]]
comparison


,n_features,mae,rmse,r2
A: baseline (numeric + categorical),38.0,248.729040,502.662966,0.651514
B: + basic sentiment,39.0,245.986092,499.837616,0.657729
C: + aspect sentiment,43.0,245.138150,498.810873,0.659979


## Step 6: Where do the aspect-sentiment features rank in importance?

In [7]:
model, cols = models["C: + aspect sentiment"]
importances = pd.Series(model.feature_importances_, index=cols)
importances.sort_values(ascending=False).head(15)


bedrooms                            0.339009
room_type_Entire home/apt           0.095917
total_reviews_count                 0.081930
bathrooms                           0.077824
room_type_Shared room               0.057948
room_type_Private room              0.031894
accommodates                        0.027155
dist_to_beach                       0.024839
latitude                            0.019248
property_type_Room in hotel         0.017627
reviews_per_month                   0.017012
number_of_reviews                   0.016126
location_safety_score               0.014300
neighbourhood_cleansed_frequency    0.014199
availability_365                    0.010393
dtype: float32